---
title: "Data shape changes under Neural Networks"
author: "Andratx Bellmunt"
abstract: >
  Short notebook to demonstrate to fellow data science colleagues how data shape changes at it evolves through
  a neural network. Inspired by the paper "Topology of deep neural networks": https://arxiv.org/abs/2004.06093.
format:
  html:
    code-fold: false
    self-contained: true
jupyter: python3
number-sections: false
---


# Initialization

## Library imports

In [ ]:
# General imports
from random import random
import numpy as np

In [ ]:
# Neural networks
import os
os.environ['TF_CPP_MIN_LOG_LEVEL'] = '3'
os.environ['TF_ENABLE_ONEDNN_OPTS'] = '0'
import tensorflow as tf
from tensorflow import keras as ks

In [ ]:
# Principal component analysis
from sklearn.decomposition import PCA

In [ ]:
# Persistent homology
from ripser import ripser

In [ ]:
# Plotting
import plotly.graph_objects as go
import plotly.io as pio
import teaspoon.TDA.Draw as Draw
import matplotlib.pyplot as plt

In [ ]:
# Set renderer for plots
pio.renderers.default = "plotly_mimetype+notebook_connected"

## Auxiliary functions and definitions

In [ ]:
relu = ks.layers.ReLU()

In [ ]:
def apply_weights(model, train_data):
    trans_train_data = [train_data]
    for i in range(len(model.layers) - 1):
        linear_part = model.layers[i].get_weights()[0]
        affine_part = model.layers[i].get_weights()[1]
        last = trans_train_data[-1]
        trans_train_data.append(relu(np.dot(last, linear_part) + affine_part))

    return np.array(trans_train_data[1:])

In [ ]:
def apply_pca(trans_train_data, dim=2):
    pca = PCA(n_components=dim)
    num_layers = trans_train_data.shape[0]
    pca_trans_train_data = np.array(
        [pca.fit_transform(trans_train_data[i]) for i in range(num_layers)]
    )
    
    return pca_trans_train_data

In [ ]:
def plot_persistent_homology(dcls, cs, num_layers):
    td = dcls[cs]["td"]
    td_diagrams = dcls[cs]["td_diagrams"]
    pttd = dcls[cs]["pttd"][num_layers-1]
    pttd_diagrams = dcls[cs]["last_pttd_diagrams"]
    
    fig, axes = plt.subplots(nrows=2, ncols=3, figsize=(20,12))
    
    # Plot data points cloud
    plt.sca(axes[0, 0])
    plt.title("Training data")
    plt.scatter(td[:,0], td[:,1], marker=".")
    
    # Plot 0-dim diagram
    plt.sca(axes[0,1])
    plt.title("0-dim diagram")
    Draw.drawDgm(td_diagrams[0])
    
    # Plot 1-dim diagram
    plt.sca(axes[0,2])
    plt.title("1-dim diagram")
    Draw.drawDgm(td_diagrams[1])
    
    # Plot data points cloud
    plt.sca(axes[1,0])
    plt.title("Transformed data")
    plt.scatter(pttd[:,0], pttd[:,1], marker=".")
    
    # Plot 0-dim diagram
    plt.sca(axes[1,1])
    plt.title("0-dim diagram")
    Draw.drawDgm(pttd_diagrams[0])
    
    # Plot 1-dim diagram
    plt.sca(axes[1,2])
    plt.title("1-dim diagram")
    try:
        Draw.drawDgm(pttd_diagrams[1])
    except:
        axes[1,2].plot([0, 1], [0, 1], transform=axes[1,2].transAxes)
        plt.axis([0,1,0,1])


# Generate sample training data

In [ ]:
dcls = {"A":{}, "B":{}}

In [ ]:
# Generate train data
train_data = []
train_labels = []
for i in range(10000):
    r = 4 * random()
    a = 2 * np.pi * random()
    x = r * np.cos(a)
    y = r * np.sin(a)
    train_data.append(np.array([x, y]))
    
    cond_left_eye = (x-np.sqrt(2))**2 + (y-np.sqrt(2))**2 < 1
    cond_right_eye = (x+np.sqrt(2))**2 + (y-np.sqrt(2))**2 < 1
    cond_mouth = (x/2)**2 + (y+2)**2 < 1
    cond_nose = x**2 + y**2 < 0.1
    train_labels.append(int(cond_left_eye or cond_right_eye or cond_mouth or cond_nose))

In [ ]:
train_data = np.array(train_data)
train_labels = np.array(train_labels)
dcls["A"]["td"] = train_data[np.where(train_labels==0)]
dcls["B"]["td"] = train_data[np.where(train_labels==1)]

In [ ]:
# Plot training data
dcls["A"]["col"] = "red"
dcls["B"]["col"] = "green"

In [ ]:
fig = go.Figure()
for cs in dcls:
    fig.add_trace(
        go.Scatter(
            x=dcls[cs]["td"][:,0], 
            y=dcls[cs]["td"][:,1],
            mode='markers',
            name=f"class_{cs}", 
            marker_color=dcls[cs]["col"],
            marker_size=2.5)
        )
fig.update_layout(
    width=600,
    height=600,
    xaxis_range=[-4.5,4.5],
    yaxis_scaleanchor="x",
    yaxis_scaleratio=1
)

fig.show()


# Create and train neural network

In [ ]:
model = ks.Sequential([
    ks.layers.Dense(15, activation="relu"),
    ks.layers.Dense(15, activation="relu"),
    ks.layers.Dense(15, activation="relu"),
    ks.layers.Dense(15, activation="relu"),
    ks.layers.Dense(15, activation="relu"),
    ks.layers.Dense(15, activation="relu"),
    ks.layers.Dense(15, activation="relu"),
    ks.layers.Dense(15, activation="relu"),
    ks.layers.Dense(15, activation="relu"),
    ks.layers.Dense(15, activation="relu"),
    ks.layers.Dense(15, activation="relu"),
    ks.layers.Dense(15, activation="relu"),
    ks.layers.Dense(15, activation="relu"),
    ks.layers.Dense(15, activation="relu"),
    ks.layers.Dense(15, activation="relu"),
    ks.layers.Dense(1, activation="sigmoid")
])

In [ ]:
model.compile(
    optimizer=ks.optimizers.Adam(),
    loss=ks.losses.BinaryCrossentropy(),
    metrics=[
        ks.metrics.BinaryAccuracy(),
        ks.metrics.FalseNegatives(),
        ks.metrics.FalsePositives()
    ]
)

In [ ]:
model.fit(train_data, train_labels, epochs=5, batch_size=1)


#### Transform data points

In [ ]:
# Transform data points
for cs in dcls:
    dcls[cs]["ttd"] = apply_weights(model, dcls[cs]["td"])

In [ ]:
# Apply PCA
for cs in dcls:
    dcls[cs]["pttd"] = apply_pca(dcls[cs]["ttd"])

In [ ]:
cs = "A"
num_layers = len(model.layers)-1

In [ ]:
fig = go.Figure()

fig.add_trace(go.Scatter(x=dcls[cs]["td"][:,0], y=dcls[cs]["td"][:,1], mode='markers', marker_color=dcls[cs]["col"], marker_size=2.5))
for i in range(num_layers):
    fig.add_trace(go.Scatter(x=dcls[cs]["pttd"][i,:,0], y=dcls[cs]["pttd"][i,:,1], mode='markers', marker_color=dcls[cs]["col"], marker_size=2.5))


In [ ]:
steps = []
for i in range(len(fig.data)):
    step = dict(
        method="update",
        args=[{"visible": [False] * len(fig.data)}],  # layout attribute
    )
    step["args"][0]["visible"][i] = True  # Toggle i'th trace to "visible"
    steps.append(step)

In [ ]:
sliders = [dict(
    active=0,
    currentvalue={"prefix": "Layer: "},
    pad={"t": 50},
    steps=steps
)]

In [ ]:
fig.update_layout(
    sliders=sliders
)

In [ ]:
fig.show()


#### Quick intro to Betti numbers (via persistent homology)


###### Betti numbers


Betti numbers encode some characteristics of shape:
- The 0th Betti number is the number of connected components of the shape.
- The 1st Betti number counts the number of loops (or holes) in the shape.
- More in general, n-th Betti number counts the number of "n-dimensional loops" in the shape.

In [ ]:
fig = go.Figure(
    go.Scatter(
        x=dcls["A"]["td"][:,0],
        y=dcls["A"]["td"][:,1],
        mode='markers',
        name=f"class_{cs}",
        marker_color=dcls["A"]["col"],
        marker_size=2.5
    )
)

fig.update_layout(
    width=600,
    height=600,
    xaxis_range=[-4.5,4.5],
    yaxis_scaleanchor="x",
    yaxis_scaleratio=1
)

fig.show()

In [ ]:
fig = go.Figure(
    go.Scatter(
        x=dcls["B"]["td"][:,0],
        y=dcls["B"]["td"][:,1],
        mode='markers',
        name=f"class_{cs}",
        marker_color=dcls["B"]["col"],
        marker_size=2.5
    )
)

fig.update_layout(
    width=600,
    height=600,
    xaxis_range=[-4.5,4.5],
    yaxis_scaleanchor="x",
    yaxis_scaleratio=1
)

fig.show()


In our example, we are in dimension 2, so the only Betti numbers that are relevant are the 0th and the 1st:
- Class A Betti numbers: (1, 4)
- Class B Betti numbers: (4, 0)


###### Persistent homology


Nice video on persistent homology: https://www.youtube.com/watch?v=SbsvM4Gcbl0

In [ ]:
for cs in ["A", "B"]:
    dcls[cs]["td_diagrams"] = ripser(dcls[cs]["td"])["dgms"]
    dcls[cs]["last_pttd_diagrams"] = ripser(dcls[cs]["pttd"][num_layers-1])["dgms"]

In [ ]:
plot_persistent_homology(dcls, "A", num_layers)

In [ ]:
plot_persistent_homology(dcls, "B", num_layers)


#### Takeaways


- Data clouds shapes can be studied through Betti numbers.
- Betti numbers can be computed with persistent homology.
- Each new layer on a neural network simplifies Betti numbers (i.e. shape).
- Once shape is simplified, it is easier to perform some tasks, such as classification.